# KNN Knowledge Base training
Initialize the frozen KB using `unknown_initial_kb.parquet` and check validation matching. Does not use final test or demo for fitting.

In [1]:
from pathlib import Path
import sys, json, shutil, time
import numpy as np
import pandas as pd
import joblib

ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="kb" and ROOT.parent.name.lower()=="notebooks":
    ROOT=ROOT.parent.parent
elif ROOT.name.lower()=="notebooks":
    ROOT=ROOT.parent
KB_DIR=ROOT/"notebooks"/"kb"
sys.path.insert(0,str(KB_DIR))
from kb_flow import read_partition, build, search, atomic_csv, update_after_ae, load_version20, score_precomputed_flow, prepare_ae_context, EXCLUDE

SPLITS=ROOT/"data/processed/Data_V3_SSH_Unknown_final/unknown_kb_splits"
KB_DATA=ROOT/"data/processed/knn_kb"
MODELS=ROOT/"saved-models/knn_kb"
RESULTS=ROOT/"outputs/results/knn_kb"
for d in (KB_DATA,MODELS,RESULTS): d.mkdir(parents=True,exist_ok=True)
INITIAL_CSV=KB_DATA/"knowledge_base.csv"
INITIAL_INDEX=MODELS/"initial_kb.joblib"
DEMO_CSV=KB_DATA/"knowledge_base_demo.csv"
DEMO_INDEX=MODELS/"demo_kb.joblib"
for name in ("unknown_initial_kb.parquet","unknown_validation.parquet",
             "unknown_final_test.parquet","unknown_demo_kb_updates.parquet"):
    if not (SPLITS/name).is_file(): raise FileNotFoundError(SPLITS/name)
print("Project:",ROOT)
print("Existing splits:",SPLITS)
print("KB outputs:",KB_DATA)

Project: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection
Existing splits: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\Data_V3_SSH_Unknown_final\unknown_kb_splits
KB outputs: C:\Users\harsh\Downloads\Projects\Hybrid-Network-Intrusion-Detection\data\processed\knn_kb


In [2]:
# 1. Initialize once. Set True ONLY when intentionally rebuilding initial KB.
REBUILD_INITIAL_KB = False
if REBUILD_INITIAL_KB or not (INITIAL_CSV.exists() and INITIAL_INDEX.exists()):
    ref=read_partition(SPLITS/"unknown_initial_kb.parquet")
    ref["pattern_id"]=[f"KB_{i:07d}" for i in range(1,len(ref)+1)]
    ref["attack_label"]=""  # No attack labels in the KB reference
    ref["source"]="initial_reference"
    ref["reviewed"]=True
    ref["ae_verified"]=False
    ref["anomaly_score"]=np.nan
    ref["date_added"]=""
    model=build(ref)
    assert not (set(c.casefold() for c in model["columns"]) & EXCLUDE), "Metadata leaked into KNN"
    if len(ref)<2: raise ValueError("Need at least two KB records for leave-one-out cutoff")
    distances,ids=model["index"].kneighbors(model["vectors"],n_neighbors=2)
    model["cutoff"]=float(np.quantile(distances[:,1],0.95))
    atomic_csv(ref,INITIAL_CSV)
    joblib.dump(model,INITIAL_INDEX)
else:
    ref=pd.read_csv(INITIAL_CSV,low_memory=False)
    model=joblib.load(INITIAL_INDEX) # Trust only your own artifacts.
if len(ref)!=model["count"]: raise ValueError("Initial KB/index count mismatch")
print("Initial KB records:",len(ref))
print("Similarity cutoff:",model["cutoff"])
validation=read_partition(SPLITS/"unknown_validation.parquet")
validation_hits=search(validation,model,ref,k=5)
print("Validation match rate:",np.mean([x["matched"] for x in validation_hits]))

Initial KB records: 5203
Similarity cutoff: 0.9453825241925351
Validation match rate: 0.9429394812680115
